# Експеримент 08: Комбінований worst-case сценарій (ExpC)

## Мета
Тестування обох задач у найгіршому сценарії:
- **Fake detection**: 50% FAKE (дисбаланс класів)
- **Topic classification**: без "новини" (4 класи)

## Модель
- Fake: Найкраща голова з Exp 02 (HeadA за замовчуванням)
- Topic: LinearSVC

In [ ]:
# Імпорт
import os
import sys
import json
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

import pymorphy3
import re

# # # # sys.path.append('..')  # Not needed  # Not needed  # Not needed  # Not needed
from utils.experiment_utils import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

In [ ]:
# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Пристрій: {device}")

# Шляхи
fake_dataset_path = r"C:\Users\igrew\OneDrive\Desktop\Course Work\Datasets\news_detector\FINAL_DATASET.csv"
topic_dataset_path = r"C:\Users\igrew\OneDrive\Desktop\Course Work\Datasets\theme_detector\ua_news_train_balanced.csv"
models_dir = 'saved_models/'
plots_dir = 'plots/'
results_dir = 'results/'

assert os.path.exists(fake_dataset_path), f"Fake датасет не знайдено"
assert os.path.exists(topic_dataset_path), f"Topic датасет не знайдено"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

## Частина 1: Fake Detection (50% FAKE)

In [ ]:
print("=" * 60)
print("ЧАСТИНА 1: FAKE DETECTION (50% FAKE)")
print("=" * 60)

# Завантаження fake датасету
df_fake = pd.read_csv(fake_dataset_path)
print(f"Повний датасет: {df_fake.shape}")

# Створення незбалансованого датасету: 100% TRUE + 50% FAKE
df_true = df_fake[df_fake['label'] == 0]
df_fake_only = df_fake[df_fake['label'] == 1]
df_fake_sampled = df_fake_only.sample(frac=0.5, random_state=SEED)
df_imbalanced = pd.concat([df_true, df_fake_sampled], ignore_index=True)
df_imbalanced = df_imbalanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Незбалансований датасет: {df_imbalanced.shape}")
print(f"Розподіл: {df_imbalanced['label'].value_counts().to_dict()}")

In [ ]:
# Підготовка даних для BERT
texts_fake = df_imbalanced['text'].values
labels_fake = df_imbalanced['label'].values

X_train_f, X_temp_f, y_train_f, y_temp_f = train_test_split(
    texts_fake, labels_fake, test_size=0.3, stratify=labels_fake, random_state=SEED
)
X_val_f, X_test_f, y_val_f, y_test_f = train_test_split(
    X_temp_f, y_temp_f, test_size=0.5, stratify=y_temp_f, random_state=SEED
)

print(f"Train: {len(X_train_f)} | Val: {len(X_val_f)} | Test: {len(X_test_f)}")

In [ ]:
# Токенізація
tokenizer_fake = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(str(self.texts[idx]), max_length=self.max_length,
                                   padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset_f = NewsDataset(X_train_f, y_train_f, tokenizer_fake)
val_dataset_f = NewsDataset(X_val_f, y_val_f, tokenizer_fake)
test_dataset_f = NewsDataset(X_test_f, y_test_f, tokenizer_fake)

train_loader_f = DataLoader(train_dataset_f, batch_size=16, shuffle=True)
val_loader_f = DataLoader(val_dataset_f, batch_size=16)
test_loader_f = DataLoader(test_dataset_f, batch_size=16)

In [ ]:
# Модель BERT HeadA
class BERTHeadA(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-multilingual-cased')
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, 2)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)

model_fake = BERTHeadA().to(device)
print("BERT модель створена")

In [ ]:
# Тренування
epochs = 3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_fake.parameters(), lr=2e-5)
total_steps = len(train_loader_f) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

print("Тренування BERT для fake detection...\n")
for epoch in range(epochs):
    model_fake.train()
    train_loss = 0
    for batch in tqdm(train_loader_f, desc=f"Epoch {epoch+1}/{epochs} - Train"):
        optimizer.zero_grad()
        outputs = model_fake(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        loss = criterion(outputs, batch['labels'].to(device))
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
    
    model_fake.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader_f, desc=f"Epoch {epoch+1}/{epochs} - Val"):
            outputs = model_fake(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss = criterion(outputs, batch['labels'].to(device))
            val_loss += loss.item()
    
    train_loss /= len(train_loader_f)
    val_loss /= len(val_loader_f)
    delta = val_loss - train_loss
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Delta: {delta:.4f}")

print("\nТренування завершено")

In [ ]:
# Оцінка
model_fake.eval()
all_preds_f = []
all_labels_f = []
all_probs_f = []

with torch.no_grad():
    for batch in tqdm(test_loader_f, desc="Eval Fake"):
        outputs = model_fake(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        all_preds_f.extend(preds.cpu().numpy())
        all_labels_f.extend(batch['labels'].numpy())
        all_probs_f.extend(probs[:, 1].cpu().numpy())

metrics_fake = {
    'accuracy': accuracy_score(all_labels_f, all_preds_f),
    'precision': precision_score(all_labels_f, all_preds_f),
    'recall': recall_score(all_labels_f, all_preds_f),
    'f1': f1_score(all_labels_f, all_preds_f),
    'roc_auc': roc_auc_score(all_labels_f, all_probs_f)
}

print("\n=== Результати Fake Detection (ExpC) ===")
for k, v in metrics_fake.items():
    print(f"{k}: {v:.4f}")

## Частина 2: Topic Classification (4 класи, без "новини")

In [ ]:
print("\n" + "=" * 60)
print("ЧАСТИНА 2: TOPIC CLASSIFICATION (4 класи)")
print("=" * 60)

# Завантаження topic датасету
df_topic = pd.read_csv(topic_dataset_path)
df_topic = df_topic[df_topic['target'] != 'новини'].copy()
df_topic.reset_index(drop=True, inplace=True)

print(f"Датасет (без 'новини'): {df_topic.shape}")
print(f"Розподіл:\n{df_topic['target'].value_counts()}")

In [ ]:
# Препроцесинг
morph = pymorphy3.MorphAnalyzer(lang='uk')
ukrainian_stopwords = set([
    'і', 'в', 'у', 'а', 'з', 'на', 'до', 'не', 'що', 'як', 'по', 'за', 'та', 'це', 'для',
    'від', 'при', 'про', 'або', 'але', 'бо', 'ж', 'же', 'й', 'ні', 'так', 'він', 'вона', 'воно',
    'вони', 'ми', 'ви', 'я', 'ти', 'його', 'її', 'їх', 'наш', 'ваш', 'їхній', 'цей', 'той', 'мій'
])

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^а-яїієґ\s]', ' ', text)
    tokens = text.split()
    lemmas = []
    for token in tokens:
        if len(token) > 2 and token not in ukrainian_stopwords:
            lemmas.append(morph.parse(token)[0].normal_form)
    return ' '.join(lemmas)

print("Препроцесинг текстів...")
df_topic['text_processed'] = [preprocess_text(t) for t in tqdm(df_topic['text'], desc="Preprocessing")]
print("Препроцесинг завершено")

In [ ]:
# Підготовка даних
X_topic = df_topic['text_processed'].values
y_topic = df_topic['target'].values

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_topic, y_topic, test_size=0.2, stratify=y_topic, random_state=SEED
)

print(f"Train: {len(X_train_t)} | Test: {len(X_test_t)}")

In [ ]:
# TF-IDF + LinearSVC
vectorizer_topic = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.9)
X_train_tfidf_t = vectorizer_topic.fit_transform(X_train_t)
X_test_tfidf_t = vectorizer_topic.transform(X_test_t)

print("Тренування LinearSVC...")
base_svm_topic = LinearSVC(random_state=SEED, max_iter=2000, dual=False)
model_topic = CalibratedClassifierCV(base_svm_topic, cv=3)
model_topic.fit(X_train_tfidf_t, y_train_t)
print("Тренування завершено")

In [ ]:
# Оцінка
y_pred_t = model_topic.predict(X_test_tfidf_t)

metrics_topic = {
    'accuracy': accuracy_score(y_test_t, y_pred_t),
    'macro_f1': f1_score(y_test_t, y_pred_t, average='macro'),
    'weighted_f1': f1_score(y_test_t, y_pred_t, average='weighted')
}

print("\n=== Результати Topic Classification (ExpC) ===")
for k, v in metrics_topic.items():
    print(f"{k}: {v:.4f}")

## Частина 3: Порівняння з baseline

In [ ]:
# Завантаження baseline результатів
with open(f'{results_dir}02_bert_fake_headA_results.json', 'r') as f:
    fake_baseline = json.load(f)
with open(f'{results_dir}04_topic_svm_base_results.json', 'r') as f:
    topic_baseline = json.load(f)

# Порівняльна таблиця для Fake
print("\n=== Порівняння Fake Detection ===")
fake_comparison = pd.DataFrame({
    'Baseline (Base)': fake_baseline,
    'ExpC (50% FAKE)': metrics_fake
}).T
print(fake_comparison)

# Порівняльна таблиця для Topic
print("\n=== Порівняння Topic Classification ===")
topic_comparison = pd.DataFrame({
    'Baseline (5 класів)': {k: topic_baseline[k] for k in ['accuracy', 'macro_f1', 'weighted_f1']},
    'ExpC (4 класи)': metrics_topic
}).T
print(topic_comparison)

In [ ]:
# Візуалізація порівняння
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Fake detection
metric_names_fake = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
baseline_vals_f = [fake_baseline[m] for m in metric_names_fake]
expC_vals_f = [metrics_fake[m] for m in metric_names_fake]

x = np.arange(len(metric_names_fake))
width = 0.35
ax1.bar(x - width/2, baseline_vals_f, width, label='Baseline', alpha=0.8)
ax1.bar(x + width/2, expC_vals_f, width, label='ExpC', alpha=0.8)
ax1.set_xlabel('Метрики')
ax1.set_ylabel('Значення')
ax1.set_title('Fake Detection: Baseline vs ExpC')
ax1.set_xticks(x)
ax1.set_xticklabels(metric_names_fake, rotation=15)
ax1.legend()
ax1.set_ylim(0, 1)
ax1.grid(True, axis='y', alpha=0.3)

# Topic classification
metric_names_topic = ['accuracy', 'macro_f1', 'weighted_f1']
baseline_vals_t = [topic_baseline[m] for m in metric_names_topic]
expC_vals_t = [metrics_topic[m] for m in metric_names_topic]

x2 = np.arange(len(metric_names_topic))
ax2.bar(x2 - width/2, baseline_vals_t, width, label='Baseline (5 класів)', alpha=0.8)
ax2.bar(x2 + width/2, expC_vals_t, width, label='ExpC (4 класи)', alpha=0.8)
ax2.set_xlabel('Метрики')
ax2.set_ylabel('Значення')
ax2.set_title('Topic Classification: Baseline vs ExpC')
ax2.set_xticks(x2)
ax2.set_xticklabels(metric_names_topic)
ax2.legend()
ax2.set_ylim(0, 1)
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{plots_dir}08_expC_baseline_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

## Збереження результатів

In [ ]:
# Збереження моделей
torch.save(model_fake.state_dict(), f'{models_dir}bert_fake_expC.pt')
with open(f'{models_dir}topic_svm_expC_model.pkl', 'wb') as f:
    pickle.dump(model_topic, f)
with open(f'{models_dir}topic_svm_expC_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer_topic, f)

# Збереження результатів
with open(f'{results_dir}08_expC_fake_results.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_fake, f, ensure_ascii=False, indent=2)

with open(f'{results_dir}08_expC_topic_results.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_topic, f, ensure_ascii=False, indent=2)

print("\n=== Результати ExpC збережено ===")
print(f"Fake model: {models_dir}bert_fake_expC.pt")
print(f"Topic model: {models_dir}topic_svm_expC_model.pkl")
print(f"Results: {results_dir}08_expC_[fake|topic]_results.json")

## Висновки

Експеримент C показує продуктивність моделей у найгіршому сценарії:
- **Fake detection**: Дисбаланс класів (2:1 TRUE:FAKE)
- **Topic classification**: Видалення найменш специфічної категорії

Порівняння з baseline дозволяє оцінити деградацію метрик та стійкість моделей до несприятливих умов.